# Analyzing gadget distance and circuit faults

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

Why can a protocol pass noiseless validation yet still fail after a single physical fault?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Explain the difference between code distance (`CodeProfile.distance`) and gadget distance (`GadgetProfile.distance`).
- Identify which baseline `bare-css/v1` gadgets drop from distance two to distance one.
- Trace single-fault witnesses and inspect changed checks, readouts, and frame signs with `effects_of`.

## Setting up

The cell below builds the validated `bare-css/v1` protocol for `C4`.

In [1]:
from qdk import ec
from _unit import make_baseline_protocol

protocol = make_baseline_protocol()
gadgets = protocol.layers[0].gadgets

## Separating protocol audits from gadget fault analysis

Our baseline `C4` protocol passed `ec.audit(protocol)` with zero errors and zero warnings, and the `C4` code itself has distance two. Does that mean every gadget tolerates (detects) any single circuit fault?

`GadgetProfile.distance()` computes the **gadget distance** (the circuit-level fault distance of the gadget): the minimum number of allowed circuit faults that changes the gadget's logical action, keeps every output inside its code space, and leaves **every declared check and flag unchanged**. Under the default circuit-level fault model, one fault after a call can apply any Pauli error on that call's qubits, flip a measurement bit it produces, or both.

Let's check the gadget distance of all six baseline gadgets:

In [2]:
print(f"{'Gadget':<16} | {'Gadget distance':<15} | {'Witness'}")
print("-" * 70)
for name, gadget in sorted(gadgets.items()):
    dist = ec.GadgetProfile(gadget).distance()
    print(f"{name:<16} | {str(dist):<15} | {dist.witness}")

Gadget           | Gadget distance | Witness
----------------------------------------------------------------------
cx_all           | 2               | X_4 after call 0; X_5 after call 1
measure_x_all    | 2               | X_0 after call 0; X_1 after call 1
measure_z_all    | 2               | flip call 0 readout 0; flip call 1 readout 0
prepare_x_all    | 1               | X_5 after call 18


prepare_z_all    | 1               | X_4 after call 7
syndrome         | 1               | X_4 after call 3


Only `cx_all`, `measure_x_all`, and `measure_z_all` achieve gadget distance `2`. Both preparations (`prepare_x_all`, `prepare_z_all`) and `syndrome` drop to **gadget distance `1`**—meaning a single physical fault inside those circuits causes an undetected logical error.

## Tracing how a single ancilla fault becomes a logical error

Why does `syndrome` have distance `1`? In `bare-css/v1`, ancilla `4` (`H`-dressed) measures $X_0 X_1 X_2 X_3$ via four sequential `CX 4 q` gates, and ancilla `5` (`H`-dressed) measures $Z_0 Z_1 Z_2 Z_3$ via four sequential `CZ 5 q` gates:

<div style="text-align:center;"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 306" width="100%" style="max-width:680px;height:auto" class="qdk-ec-fault" role="img" data-asset="tutorial_fault_spreading.svg" aria-label="Six-wire bare syndrome extraction circuit showing X_4 after CX 4 1 spreading to X_2 X_3, which conjugates through CZ 5 2 and CZ 5 3 to two canceling Z phase flips on ancilla q5, leaving the Z check unflipped."> <style> .qdk-ec-fault { --qdk-fg: var(--vscode-editor-foreground, var(--jp-content-font-color1, currentColor)); --qdk-muted: color-mix(in srgb, var(--qdk-fg) 80%, transparent); --qdk-border: color-mix(in srgb, var(--qdk-fg) 30%, transparent); font-family: "Segoe UI", -apple-system, BlinkMacSystemFont, Roboto, Helvetica, Arial, sans-serif; } .qdk-ec-fault .cluster-blue { fill: color-mix(in srgb, #1976d2 8%, transparent); stroke: color-mix(in srgb, #1976d2 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-fault .cluster-teal { fill: color-mix(in srgb, #0d9488 8%, transparent); stroke: color-mix(in srgb, #0d9488 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-fault .cluster-purple { fill: color-mix(in srgb, #7c3aed 8%, transparent); stroke: color-mix(in srgb, #7c3aed 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-fault .cluster-neutral { fill: color-mix(in srgb, var(--qdk-fg) 4%, transparent); stroke: color-mix(in srgb, var(--qdk-fg) 32%, transparent); stroke-width: 1.8; } .qdk-ec-fault .cluster-red { fill: color-mix(in srgb, #dc2626 9%, transparent); stroke: color-mix(in srgb, #dc2626 62%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-fault .stage-blue { fill: color-mix(in srgb, #1976d2 18%, transparent); stroke: color-mix(in srgb, #1976d2 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-fault .stage-teal { fill: color-mix(in srgb, #0d9488 18%, transparent); stroke: color-mix(in srgb, #0d9488 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-fault .stage-purple { fill: color-mix(in srgb, #7c3aed 18%, transparent); stroke: color-mix(in srgb, #7c3aed 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-fault .stage-red { fill: color-mix(in srgb, #dc2626 16%, transparent); stroke: color-mix(in srgb, #dc2626 78%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-fault .loop-x { fill: color-mix(in srgb, #1976d2 20%, transparent); stroke: color-mix(in srgb, #1976d2 85%, var(--qdk-fg)); stroke-width: 2.2; } .qdk-ec-fault .loop-z { fill: color-mix(in srgb, #0d9488 20%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2.2; stroke-dasharray: 6 3; } .qdk-ec-fault .hdr-blue { fill: color-mix(in srgb, #1976d2 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-fault .hdr-teal { fill: color-mix(in srgb, #0d9488 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-fault .hdr-purple { fill: color-mix(in srgb, #7c3aed 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-fault .hdr-neutral { fill: var(--qdk-fg); font-size: 12.2px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-fault .hdr-red { fill: color-mix(in srgb, #dc2626 75%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-fault .box-title { fill: var(--qdk-fg); font-size: 11.6px; font-weight: 700; } .qdk-ec-fault .box-sub { fill: var(--qdk-muted); font-size: 10px; } .qdk-ec-fault .code-tag { fill: var(--qdk-fg); font-family: Consolas, "Courier New", monospace; font-size: 9.4px; font-weight: 600; } .qdk-ec-fault .badge-blue { fill: #1976d2; } .qdk-ec-fault .badge-teal { fill: #0d9488; } .qdk-ec-fault .badge-purple { fill: #7c3aed; } .qdk-ec-fault .badge-red { fill: #dc2626; } .qdk-ec-fault .badge-txt { fill: #ffffff; font-size: 9.6px; font-weight: 700; } .qdk-ec-fault .conn { stroke: color-mix(in srgb, var(--qdk-fg) 68%, transparent); stroke-width: 2.2; fill: none; } .qdk-ec-fault .wire { stroke: color-mix(in srgb, var(--qdk-fg) 48%, transparent); stroke-width: 1.5; fill: none; } .qdk-ec-fault .gate-neutral { stroke: color-mix(in srgb, var(--qdk-fg) 78%, transparent); stroke-width: 1.8; fill: none; } .qdk-ec-fault .gate-red { stroke: #dc2626; stroke-width: 2.3; fill: none; } .qdk-ec-fault .gate-teal { stroke: #0d9488; stroke-width: 2.4; fill: none; } .qdk-ec-fault .qubit-node { fill: var(--vscode-editor-background, #1e1e1e); stroke: color-mix(in srgb, var(--qdk-fg) 75%, transparent); stroke-width: 2; } .qdk-ec-fault .anc-node { fill: color-mix(in srgb, #0d9488 22%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2; } </style> <rect x="12" y="12" width="656" height="282" rx="10" class="cluster-red"/> <text x="340" y="34" text-anchor="middle" class="hdr-red">BARE SYNDROME SCHEDULE &#8212; SPREADING FAULT UNDETECTED (GADGET DISTANCE = 1)</text> <text x="28" y="66" class="code-tag">q0</text> <line x1="78" y1="62" x2="572" y2="62" class="wire"/> <text x="28" y="90" class="code-tag">q1</text> <line x1="78" y1="86" x2="572" y2="86" class="wire"/> <text x="28" y="114" class="code-tag">q2</text> <line x1="78" y1="110" x2="572" y2="110" class="wire"/> <text x="28" y="138" class="code-tag">q3</text> <line x1="78" y1="134" x2="572" y2="134" class="wire"/> <text x="24" y="168" class="code-tag">q4 (X)</text> <line x1="78" y1="164" x2="572" y2="164" class="wire"/> <text x="24" y="198" class="code-tag">q5 (Z)</text> <line x1="78" y1="194" x2="572" y2="194" class="wire"/> <rect x="86" y="154" width="22" height="20" rx="4" class="qubit-node"/> <text x="97" y="168" text-anchor="middle" class="code-tag">H</text> <line x1="126" y1="164" x2="126" y2="62" class="gate-neutral"/> <circle cx="126" cy="164" r="4" fill="var(--qdk-fg)"/> <circle cx="126" cy="62" r="7" class="qubit-node"/> <text x="126" y="66" text-anchor="middle" class="code-tag">+</text> <line x1="162" y1="164" x2="162" y2="86" class="gate-neutral"/> <circle cx="162" cy="164" r="4" fill="var(--qdk-fg)"/> <circle cx="162" cy="86" r="7" class="qubit-node"/> <text x="162" y="90" text-anchor="middle" class="code-tag">+</text> <rect x="184" y="154" width="44" height="20" rx="4" class="badge-red"/> <text x="206" y="168" text-anchor="middle" class="badge-txt">X_4</text> <line x1="250" y1="164" x2="250" y2="110" class="gate-red"/> <circle cx="250" cy="164" r="4.5" fill="#dc2626"/> <circle cx="250" cy="110" r="7" class="qubit-node"/> <text x="250" y="114" text-anchor="middle" class="code-tag">+</text> <line x1="286" y1="164" x2="286" y2="134" class="gate-red"/> <circle cx="286" cy="164" r="4.5" fill="#dc2626"/> <circle cx="286" cy="134" r="7" class="qubit-node"/> <text x="286" y="138" text-anchor="middle" class="code-tag">+</text> <rect x="308" y="154" width="22" height="20" rx="4" class="qubit-node"/> <text x="319" y="168" text-anchor="middle" class="code-tag">H</text> <rect x="342" y="184" width="22" height="20" rx="4" class="qubit-node"/> <text x="353" y="198" text-anchor="middle" class="code-tag">H</text> <line x1="384" y1="62" x2="384" y2="194" class="gate-neutral"/> <circle cx="384" cy="62" r="4.2" fill="var(--qdk-fg)"/> <circle cx="384" cy="194" r="4.2" fill="var(--qdk-fg)"/> <line x1="420" y1="86" x2="420" y2="194" class="gate-neutral"/> <circle cx="420" cy="86" r="4.2" fill="var(--qdk-fg)"/> <circle cx="420" cy="194" r="4.2" fill="var(--qdk-fg)"/> <line x1="460" y1="110" x2="460" y2="194" class="gate-red" stroke-dasharray="4 2"/> <circle cx="460" cy="110" r="4.5" fill="#dc2626"/> <circle cx="460" cy="194" r="4.5" fill="#dc2626"/> <line x1="500" y1="134" x2="500" y2="194" class="gate-red" stroke-dasharray="4 2"/> <circle cx="500" cy="134" r="4.5" fill="#dc2626"/> <circle cx="500" cy="194" r="4.5" fill="#dc2626"/> <rect x="524" y="184" width="22" height="20" rx="4" class="qubit-node"/> <text x="535" y="198" text-anchor="middle" class="code-tag">H</text> <rect x="578" y="101" width="74" height="18" rx="4" class="badge-red"/> <text x="615" y="114" text-anchor="middle" class="badge-txt">data: X_2</text> <rect x="578" y="125" width="74" height="18" rx="4" class="badge-red"/> <text x="615" y="138" text-anchor="middle" class="badge-txt">data: X_3</text> <rect x="578" y="185" width="74" height="18" rx="4" class="stage-red"/> <text x="615" y="198" text-anchor="middle" class="code-tag">&#916;check = 0</text> <rect x="28" y="220" width="624" height="58" rx="7" class="stage-red"/> <text x="340" y="241" text-anchor="middle" class="box-title">X_2 and X_3 each conjugate across CZ to Z on q5 &#8594; two phase flips cancel (&#916;M5 = 0)</text> <text x="340" y="262" text-anchor="middle" class="code-tag">Leaves weight-2 logical error X_2 X_3 (flips out[0].z[0]) &#183; checks = []</text> </svg></div>

<div style="text-align:center;">

*In the unflagged sequential `bare-css/v1` schedule, a midpoint $X_4$ fault after `CX 4 1` copies onto `q2` and `q3`; those two $X$ errors each flip ancilla `5`'s $Z$ phase once ($1 \oplus 1 = 0$), leaving every check unchanged while flipping $\bar{Z}_0$ via $X_2 X_3$.*

</div>

We can inspect every distance-one witness directly in Python using `distance.witnesses` and `profile.effects_of(...)`, or probe any specific fault location with `ec.FaultEvent.after(call_index, pauli)`:

In [3]:
syn_profile = ec.GadgetProfile(gadgets["syndrome"])
calls = list(gadgets["syndrome"].circuit.calls())
syn_distance = syn_profile.distance()

faults = [w.product for w in syn_distance.witnesses]
for fault, effect in zip(faults, syn_profile.effects_of(faults)):
    loc = fault.locations[0]
    call = calls[loc.after_call]
    call_desc = f"{call.mnemonic} {list(call.operands)}"
    print(f"{fault}  (after call {loc.after_call}: {call_desc})  ->  {effect}")

# Compare against a detectable single-qubit data fault after call 0
harmless = ec.FaultEvent.after(0, ec.Pauli("X_0"))
print("\nSingle data fault:", harmless, "->", syn_profile.effects_of([harmless])[0])

X_4 after call 3  (after call 3: CX [4, 1])  ->  ['out[0].z[0]']


X_2 X_4 after call 4  (after call 4: CX [4, 2])  ->  ['out[0].z[0]']
X_5 after call 10  (after call 10: CZ [5, 1])  ->  ['out[0].x[1]']
Z_2 X_5 after call 11  (after call 11: CZ [5, 2])  ->  ['out[0].x[1]']

Single data fault: X_0 after call 0 -> ['checks[1]', 'out[0].z[0]', 'out[0].z[1]', 'out[0].stabilizers[1]']


Compare the two outputs:
- Two examples from the distance-one witnesses are `X_4` after `CX 4 1` and `X_5` after `CZ 5 1`. The first spreads to $X_2 X_3$; the second propagates through `CZ 5 2` and `CZ 5 3` as $Z_2 Z_3$ on the data, while `H 5` turns the ancilla fault into $Z_5$ before `M 5`. Each flips an output logical frame (`out[0].z[0]` or `out[0].x[1]`) with **zero `checks` changed**.
- By contrast, the single data fault `X_0` after call `0` flips `checks[1]`, alerting the decoder.

The `C4` code itself already detects every weight-one data error; in the next unit, we redesign the vulnerable gadget circuits—not the code—so no single fault leaves every check and flag unchanged.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does an X fault on ancilla 4 halfway through bare syndrome extraction leave ancilla 5's Z check unchanged?</summary>

<div style="padding:0.1em 1em;">

An $X$ error on the control of a `CX` gate copies onto its target, so `X_4` after `CX 4 1` spreads through `CX 4 2` and `CX 4 3` onto data qubits `2` and `3`. Because `X_2 X_3` anticommutes with $Z$ on **two** data qubits, its two phase kicks on ancilla `5` cancel ($1 \oplus 1 = 0$), producing no check flip.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How does effects_of distinguish a detectable fault from a distance-reducing fault?</summary>

<div style="padding:0.1em 1em;">

A detectable fault changes at least one entry in `effect.checks` (or `effect.readouts` for a flag), whereas a distance-reducing witness changes logical outputs (`effect.frames` or observable readouts) while leaving every check and flag unchanged.

</div>

</details></div>

## Exercise: Compute baseline syndrome gadget distance

Complete `baseline_syndrome_distance` so it computes and returns the integer gadget distance of the unflagged `bare-css/v1` `syndrome` gadget.

In [ ]:
from qdk import ec
from _unit import exercise, make_baseline_protocol


@exercise
def baseline_syndrome_distance() -> int:
    syn = make_baseline_protocol().layers[0].gadgets['syndrome']
    # Return the integer gadget distance of syn using ec.GadgetProfile
    return ...

**Hint**

Gadget distance is the minimum number of circuit faults that evade all declared checks and flags. Call `ec.GadgetProfile(syn).distance().value`.

In [ ]:
from qdk import ec
from _unit import exercise, make_baseline_protocol


@exercise
def baseline_syndrome_distance() -> int:
    syn = make_baseline_protocol().layers[0].gadgets['syndrome']
    return ec.GadgetProfile(syn).distance().value

`1`. The four `CX` gates for the $X$ stabilizer and the four `CZ` gates for the $Z$ stabilizer run sequentially, without interleaving or flags. A midpoint ancilla fault can spread to a weight-two logical error on the data block without flipping any check.

## Further reading

- [Analyze gadget faults ↗](https://learn.microsoft.com/en-us/azure/quantum/develop-validate-qec-protocol#analyze-gadget-faults)
- [Separate protocol validation from fault analysis ↗](https://learn.microsoft.com/en-us/azure/quantum/qdk-error-correction-overview#separate-protocol-validation-from-fault-analysis)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../04-validate-and-audit/validate_and_audit.workbook.ipynb" title="Previous unit: Validating and Auditing a Protocol" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Validating and Auditing a Protocol</span></span></a><a href="../06-fault-tolerant-gadgets/fault_tolerant_gadgets.workbook.ipynb" title="Next unit: Upgrading Gadgets for Fault Tolerance" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Upgrading Gadgets for Fault Tolerance</span></span><span aria-hidden="true">&#8250;</span></a></div>